# 03 — Retrieval tuning summary

End-to-end retrieval pipeline summary after d4–d7 tuning:

1. **Dense** — `bge-m3` + Qdrant HNSW `m=16, ef_construct=200`
2. **Sparse** — `rank_bm25` with tuned `k1=0.8, b=0.5`
3. **Fusion** — alpha-weighted with tuned `alpha=0.3`
4. **Reranker** — `bge-reranker-v2-m3` cross-encoder, `top_20 → top_5`

All numbers come from `evaluation/reports/retrieval_report.md` (regenerated
by `scripts/eval_retrieval.py --rerank`).

In [1]:
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
REPORT = ROOT / 'evaluation' / 'reports' / 'retrieval_report.md'

report_text = REPORT.read_text()
print(report_text.splitlines()[0])

# Retrieval evaluation — `golden_set_v1`


## 1. Overall metrics — dense → sparse → hybrid → reranker


In [2]:
overall = pd.DataFrame(
    [
        ('dense (bge-m3)',     0.7261, 0.6855, 0.8600, 0.7600, 1.1),
        ('sparse (BM25 tuned)', 0.7844, 0.7532, 0.8800, 0.8600, 0.2),
        ('hybrid RRF (k=60)',  0.7783, 0.7513, 0.8600, 0.8600, 1.3),
        ('hybrid alpha=0.3',   0.7890, 0.7589, 0.8800, 0.8600, 1.3),
        ('hybrid + reranker',  0.8089, 0.7933, 0.8600, 0.8400, 4.1),
    ],
    columns=['method', 'nDCG@10', 'MAP', 'Recall@10', 'Recall@5', 'latency_s'],
)
overall

,method,nDCG@10,MAP,Recall@10,Recall@5,latency_s
0,dense (bge-m3),0.7261,0.6855,0.86,0.76,1.1
1,sparse (BM25 tuned),0.7844,0.7532,0.88,0.86,0.2
2,hybrid RRF (k=60),0.7783,0.7513,0.86,0.86,1.3
3,hybrid alpha=0.3,0.7890,0.7589,0.88,0.86,1.3
4,hybrid + reranker,0.8089,0.7933,0.86,0.84,4.1


## 2. Per-category breakdown

Where the reranker helps the most.


In [3]:
factual = pd.DataFrame(
    [
        ('dense',             0.8982, 0.8662, 1.0000, 0.9500),
        ('sparse',            0.9004, 0.8688, 1.0000, 0.9500),
        ('hybrid RRF',        0.9631, 0.9500, 1.0000, 1.0000),
        ('hybrid alpha',      0.9346, 0.9125, 1.0000, 1.0000),
        ('hybrid + reranker', 0.9601, 0.9479, 1.0000, 1.0000),
    ],
    columns=['method', 'nDCG@10', 'MAP', 'Recall@10', 'Recall@5'],
)
tool_usage = pd.DataFrame(
    [
        ('dense',             0.3650, 0.3152, 0.5333, 0.4000),
        ('sparse',            0.4386, 0.3856, 0.6000, 0.6000),
        ('hybrid RRF',        0.3929, 0.3489, 0.5333, 0.5333),
        ('hybrid alpha',      0.4083, 0.3463, 0.6000, 0.5333),
        ('hybrid + reranker', 0.4841, 0.4708, 0.5333, 0.5333),
    ],
    columns=['method', 'nDCG@10', 'MAP', 'Recall@10', 'Recall@5'],
)
multi_hop = pd.DataFrame(
    [
        ('dense',             0.8576, 0.8148, 1.0000, 0.8667),
        ('sparse',            0.9754, 0.9667, 1.0000, 1.0000),
        ('hybrid RRF',        0.9175, 0.8889, 1.0000, 1.0000),
        ('hybrid alpha',      0.9754, 0.9667, 1.0000, 1.0000),
        ('hybrid + reranker', 0.9706, 0.9583, 1.0000, 1.0000),
    ],
    columns=['method', 'nDCG@10', 'MAP', 'Recall@10', 'Recall@5'],
)
print('factual (20 queries):')
display(factual)
print('\ntool_usage (15 queries — the hard category):')
display(tool_usage)
print('\nmulti_hop (15 queries):')
display(multi_hop)

factual (20 queries):


,method,nDCG@10,MAP,Recall@10,Recall@5
0,dense,0.8982,0.8662,1.0,0.95
1,sparse,0.9004,0.8688,1.0,0.95
2,hybrid RRF,0.9631,0.9500,1.0,1.00
3,hybrid alpha,0.9346,0.9125,1.0,1.00
4,hybrid + reranker,0.9601,0.9479,1.0,1.00



tool_usage (15 queries — the hard category):


,method,nDCG@10,MAP,Recall@10,Recall@5
0,dense,0.3650,0.3152,0.5333,0.4000
1,sparse,0.4386,0.3856,0.6000,0.6000
2,hybrid RRF,0.3929,0.3489,0.5333,0.5333
3,hybrid alpha,0.4083,0.3463,0.6000,0.5333
4,hybrid + reranker,0.4841,0.4708,0.5333,0.5333



multi_hop (15 queries):


,method,nDCG@10,MAP,Recall@10,Recall@5
0,dense,0.8576,0.8148,1.0,0.8667
1,sparse,0.9754,0.9667,1.0,1.0000
2,hybrid RRF,0.9175,0.8889,1.0,1.0000
3,hybrid alpha,0.9754,0.9667,1.0,1.0000
4,hybrid + reranker,0.9706,0.9583,1.0,1.0000


## 3. Reranker delta — does it earn its 2.8s of latency?

We compare `hybrid alpha=0.3` (the best non-rerank baseline) against
`hybrid + reranker` per category.


In [4]:
deltas = pd.DataFrame(
    [
        ('overall',    0.8089 - 0.7890, 0.7933 - 0.7589, 0.8600 - 0.8800),
        ('factual',    0.9601 - 0.9346, 0.9479 - 0.9125, 0.0),
        ('tool_usage', 0.4841 - 0.4083, 0.4708 - 0.3463, 0.5333 - 0.6000),
        ('multi_hop',  0.9706 - 0.9754, 0.9583 - 0.9667, 0.0),
    ],
    columns=['category', 'd nDCG@10', 'd MAP', 'd Recall@10'],
)
deltas.style.format({'d nDCG@10': '{:+.4f}', 'd MAP': '{:+.4f}', 'd Recall@10': '{:+.4f}'})

,category,d nDCG@10,d MAP,d Recall@10
0,overall,+0.0199,+0.0344,-0.0200
1,factual,+0.0255,+0.0354,+0.0000
2,tool_usage,+0.0758,+0.1245,-0.0667
3,multi_hop,-0.0048,-0.0084,+0.0000


**Reading.**

- **Overall:** +0.020 nDCG, +0.034 MAP, but −0.020 Recall@10. The reranker
  trades a couple of relevant chunks at the bottom of the list for sharper
  ordering at the top — exactly what nDCG and MAP reward.
- **factual:** +0.026 nDCG, +0.035 MAP, Recall stable. Solid win.
- **tool_usage:** +0.076 nDCG, **+0.124 MAP**, but −0.067 Recall@10. The
  large MAP gain confirms the cross-encoder is much better at telling apart
  near-duplicate `nmap`/`nuclei` chunks where bi-encoders fail.
- **multi_hop:** essentially break-even. The pre-rerank pipeline already
  scored Recall@10 = 1.0, so there is nothing left for the reranker to fix.

**Verdict.** Reranker is **kept** in the MVP pipeline because the MAP gain on
`tool_usage` is too large to ignore. The 2.8 s latency cost is well within
the NFR target (`p95 ≤ 8 s` end-to-end including LLM generation).

## 4. Final tuned configuration


In [5]:
import sys
sys.path.insert(0, str(ROOT))

from app.config import settings  # noqa: E402

config = pd.Series(
    {
        'embedder':           settings.bge_m3_model,
        'qdrant_collection':  settings.qdrant_collection,
        'bm25_k1':            settings.bm25_k1,
        'bm25_b':             settings.bm25_b,
        'fusion_method':      settings.fusion_method,
        'fusion_alpha':       settings.fusion_alpha,
        'rrf_k':              settings.rrf_k,
        'reranker':           settings.reranker_model,
        'retrieve_top_k':     settings.retrieve_top_k,
        'rerank_top_k':       settings.rerank_top_k,
    },
    name='value',
)
config

embedder                         BAAI/bge-m3
qdrant_collection                   garag_v1
bm25_k1                                  0.8
bm25_b                                   0.5
fusion_method                          alpha
fusion_alpha                             0.3
rrf_k                                     60
reranker             BAAI/bge-reranker-v2-m3
retrieve_top_k                            20
rerank_top_k                               5
Name: value, dtype: object